# Prophet (classical baseline)

Prophet is a classical, per-series model (trend + yearly seasonality). One model is
fit per series, so it does not scale to thousands of series. Following the guidance not
to spend much time on classical models, we fit it on a sample of series and report the
WMAE on those series' real validation rows.

Logged to the shared DagsHub. This is a reference point, not directly comparable to the
global models (which score over all series).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import logging
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.models.prophet_pipeline import build_prophet
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

# Shared DagsHub tracking.
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Prophet_Training")

# Load the model config.
with open("configs/prophet_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/09 22:23:51 INFO mlflow.tracking.fluent: Experiment with name 'Prophet_Training' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

train: (421570, 5)


## Run 1: Data Preparation

In [4]:
with mlflow.start_run(run_name="Prophet_Data_Prep"):
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, split_date)
    real_valid = get_real_validation(train_raw, stores, features, split_date)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", split_date)
    mlflow.log_param("sample_size", params["sample_size"])
    mlflow.log_param("horizon_weeks", int(horizon))

    print("series:", Y_df["unique_id"].nunique(), "| horizon:", horizon)

series: 3331 | horizon: 43
🏃 View run Prophet_Data_Prep at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/8/runs/5ae868bfec3a461c819ae7ff4ef7b1a6
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/8


## Run 2: Prophet on a sample of series

Fit one Prophet per sampled series, forecast the validation horizon, and score against
each series' real validation rows with the shared `calculate_wmae`.

In [5]:
# Pick eligible series: enough history and present in the real validation.
train_counts = train_df.groupby("unique_id").size()
valid_ids = set(real_valid["unique_id"].unique())
eligible = [uid for uid in train_counts.index if train_counts[uid] >= 60 and uid in valid_ids]

rng = np.random.default_rng(SEED)
sample_ids = list(rng.choice(eligible, size=min(params["sample_size"], len(eligible)), replace=False))
print("eligible:", len(eligible), "| sampled:", len(sample_ids))

eligible: 3110 | sampled: 30


In [6]:
with mlflow.start_run(run_name="Prophet_Subset"):
    mlflow.log_params(params)

    all_true, all_pred, all_hol = [], [], []
    for i, uid in enumerate(sample_ids, 1):
        tr = train_df[train_df["unique_id"] == uid][["ds", "y"]].sort_values("ds")
        m = build_prophet(params)
        m.fit(tr)

        future = m.make_future_dataframe(periods=horizon, freq=FREQ)
        fc = m.predict(future)[["ds", "yhat"]]

        va = real_valid[real_valid["unique_id"] == uid][["ds", "y", "IsHoliday"]]
        merged = va.merge(fc, on="ds", how="inner")
        all_true.extend(merged["y"].tolist())
        all_pred.extend(merged["yhat"].clip(lower=0).tolist())
        all_hol.extend(merged["IsHoliday"].tolist())
        if i % 10 == 0:
            print(f"  fitted {i}/{len(sample_ids)} series")

    wmae = calculate_wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))
    mlflow.log_metric("WMAE", float(wmae))
    print(f"Prophet subset WMAE ({len(sample_ids)} series, real validation): {wmae:,.2f}")

22:24:18 - cmdstanpy - INFO - Chain [1] start processing
22:24:18 - cmdstanpy - INFO - Chain [1] done processing
22:24:19 - cmdstanpy - INFO - Chain [1] start processing
22:24:19 - cmdstanpy - INFO - Chain [1] done processing
22:24:19 - cmdstanpy - INFO - Chain [1] start processing
22:24:19 - cmdstanpy - INFO - Chain [1] done processing
22:24:19 - cmdstanpy - INFO - Chain [1] start processing
22:24:19 - cmdstanpy - INFO - Chain [1] done processing
22:24:19 - cmdstanpy - INFO - Chain [1] start processing
22:24:19 - cmdstanpy - INFO - Chain [1] done processing
22:24:19 - cmdstanpy - INFO - Chain [1] start processing
22:24:20 - cmdstanpy - INFO - Chain [1] done processing
22:24:20 - cmdstanpy - INFO - Chain [1] start processing
22:24:20 - cmdstanpy - INFO - Chain [1] done processing
22:24:20 - cmdstanpy - INFO - Chain [1] start processing
22:24:20 - cmdstanpy - INFO - Chain [1] done processing
22:24:20 - cmdstanpy - INFO - Chain [1] start processing
22:24:20 - cmdstanpy - INFO - Chain [1]

  fitted 10/30 series


22:24:21 - cmdstanpy - INFO - Chain [1] start processing
22:24:21 - cmdstanpy - INFO - Chain [1] done processing
22:24:21 - cmdstanpy - INFO - Chain [1] start processing
22:24:21 - cmdstanpy - INFO - Chain [1] done processing
22:24:21 - cmdstanpy - INFO - Chain [1] start processing
22:24:21 - cmdstanpy - INFO - Chain [1] done processing
22:24:22 - cmdstanpy - INFO - Chain [1] start processing
22:24:22 - cmdstanpy - INFO - Chain [1] done processing
22:24:22 - cmdstanpy - INFO - Chain [1] start processing
22:24:22 - cmdstanpy - INFO - Chain [1] done processing
22:24:22 - cmdstanpy - INFO - Chain [1] start processing
22:24:22 - cmdstanpy - INFO - Chain [1] done processing
22:24:22 - cmdstanpy - INFO - Chain [1] start processing
22:24:22 - cmdstanpy - INFO - Chain [1] done processing
22:24:23 - cmdstanpy - INFO - Chain [1] start processing
22:24:23 - cmdstanpy - INFO - Chain [1] done processing
22:24:23 - cmdstanpy - INFO - Chain [1] start processing
22:24:23 - cmdstanpy - INFO - Chain [1]

  fitted 20/30 series


22:24:23 - cmdstanpy - INFO - Chain [1] start processing
22:24:23 - cmdstanpy - INFO - Chain [1] done processing
22:24:24 - cmdstanpy - INFO - Chain [1] start processing
22:24:24 - cmdstanpy - INFO - Chain [1] done processing
22:24:24 - cmdstanpy - INFO - Chain [1] start processing
22:24:24 - cmdstanpy - INFO - Chain [1] done processing
22:24:24 - cmdstanpy - INFO - Chain [1] start processing
22:24:24 - cmdstanpy - INFO - Chain [1] done processing
22:24:24 - cmdstanpy - INFO - Chain [1] start processing
22:24:24 - cmdstanpy - INFO - Chain [1] done processing
22:24:24 - cmdstanpy - INFO - Chain [1] start processing
22:24:24 - cmdstanpy - INFO - Chain [1] done processing
22:24:25 - cmdstanpy - INFO - Chain [1] start processing
22:24:25 - cmdstanpy - INFO - Chain [1] done processing
22:24:25 - cmdstanpy - INFO - Chain [1] start processing
22:24:25 - cmdstanpy - INFO - Chain [1] done processing
22:24:25 - cmdstanpy - INFO - Chain [1] start processing
22:24:25 - cmdstanpy - INFO - Chain [1]

  fitted 30/30 series
Prophet subset WMAE (30 series, real validation): 1,575.28
🏃 View run Prophet_Subset at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/8/runs/2640a4c007084f6c9589ab40949aead0
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/8


## Notes

- WMAE is on the real validation rows of a 30-series sample, so it is a rough reference
  and not directly comparable to the global models scored over all series.
- Params live in `configs/prophet_config.yaml`; the model is built in
  `src/models/prophet_pipeline.py`.
- Classical models fit one model per series and cannot share information across series.